# Notebook 4 — Démographie, modélisation et prédictions 2027

## Contexte
Ce notebook présente :
1. L'analyse démographique fine par tranche d'âge
2. La **modélisation ML** pour prédire l'abstention en 2027
3. L'évaluation des modèles (validation Leave-One-Year-Out)
4. Les **3 scénarios de projection** pour 2027

⚠️ **Note méthodologique** : avec seulement 6 années électorales (1995–2022), les modèles prédictifs sont nécessairement limités. On adopte une validation rigoureuse (LOYO) et on privilégie la **transparence des incertitudes** sur la précision.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score, silhouette_score
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path('..').resolve()
OUT_DIR  = BASE_DIR / 'outputs'

df_demo = pd.read_csv(OUT_DIR / 'elections_with_demography.csv')
df_cand = pd.read_csv(OUT_DIR / 'elections_candidats.csv')
print('Données chargées :', df_demo.shape)


## 1. Analyse démographique par tranche d'âge

On décompose la population en 5 tranches d'âge (0-19, 20-39, 40-59, 60-74, 75+) pour mesurer leur corrélation individuelle avec l'abstention.


In [ ]:
df_t1 = df_demo[(df_demo['tour']==1) & df_demo['pop_ens_total'].notna()].copy()
pop   = df_t1['pop_ens_total'].replace(0, np.nan)

groupes = {'0–19 ans':('pop_ens_0_19','#2ecc71'),
           '20–39 ans':('pop_ens_20_39','#27ae60'),
           '40–59 ans':('pop_ens_40_59','#3498db'),
           '60–74 ans':('pop_ens_60_74','#e67e22'),
           '75 ans +':('pop_ens_75p','#e74c3c')}

corr_results = {}
for label,(col,_) in groupes.items():
    if col in df_t1.columns:
        pct = df_t1[col] / pop * 100
        r = pct.corr(df_t1['taux_abstention'])
        corr_results[label] = r

print('Corrélations par tranche d\'âge :')
for k,v in corr_results.items():
    sens = '↑ plus d\'abstention' if v>0 else '↓ moins d\'abstention'
    print(f'  {k:12s} r={v:+.3f}  {sens}')


## 2. Feature engineering

On construit un dataset `(département × année)` avec :
- Features démographiques : `pct_jeunes`, `pct_actifs`, `pct_seniors`
- Features de tendance électorale : abstention de l'élection précédente (`lag_taux_abstention`)
- Features politiques : scores des familles au T1 de l'élection précédente

**Règle anti-fuite de données** : on n'utilise que des données connues avant l'élection cible.


In [ ]:
reg_data = pd.read_csv(OUT_DIR / 'outputs' / 'tables' / 'regression_dataset.csv'
                       if (OUT_DIR / 'outputs').exists()
                       else OUT_DIR / 'tables' / 'regression_dataset.csv')
print('Dataset de régression :', reg_data.shape)
print('Colonnes :', reg_data.columns.tolist())
print('\nAnnées disponibles :', sorted(reg_data['annee'].unique()))
print('Extrait :')
reg_data.head(3)


## 3. Validation Leave-One-Year-Out (LOYO)

Avec seulement 6 années, la validation croisée standard n'est pas adaptée. On utilise la **LOYO** : pour chaque année de test, on entraîne sur toutes les autres années.

⚠️ **Limite connue** : les R² sont négatifs pour plusieurs combinaisons, ce qui montre que la généralisation inter-années est difficile. Cela est attendu : le comportement électoral d'une année dépend fortement du contexte politique (candidats, enjeux), difficile à capturer avec des features démographiques seules.


In [ ]:
loyo = pd.read_csv(OUT_DIR / 'tables' / 'loyo_cv_results.csv')
print('Résultats LOYO :')
print(loyo.to_string(index=False))

# Synthèse par modèle
print('\nRMSE moyen par modèle :')
print(loyo.groupby('model')['RMSE'].mean().round(3))
print('\nR² moyen par modèle :')
print(loyo.groupby('model')['R²'].mean().round(3))

print('\n→ Ridge est le plus stable, GradientBoosting le plus performant récemment (2017,2022)')


In [ ]:
# Visualisation des résultats LOYO
fig, axes = plt.subplots(1,2,figsize=(12,5))
models = loyo['model'].unique()
colors = {'Ridge':'#1f77b4','RandomForest':'#ff7f0e','GradientBoosting':'#2ca02c'}

for ax, metric, ylbl in zip(axes,['RMSE','R²'],['RMSE (points de %)','R²']):
    for m in models:
        sub = loyo[loyo.model==m].sort_values('year')
        ax.plot(sub['year'], sub[metric], marker='o', label=m, color=colors.get(m,'grey'))
    if metric=='R²': ax.axhline(0, color='red', linestyle='--', linewidth=1, label='Référence (moyenne)')
    ax.set_title(f'{metric} par année de test (LOYO)')
    ax.set_xlabel('Année de test') ; ax.set_ylabel(ylbl)
    ax.legend(fontsize=8) ; ax.grid(alpha=0.3)
plt.tight_layout() ; plt.show()


## 4. Clustering des départements (métropole uniquement)

On identifie des **profils de départements** basés sur leur comportement électoral et leur structure démographique. On exclut les DOM pour éviter un clustering trivial (DOM vs métropole).


In [ ]:
clusters = pd.read_csv(OUT_DIR / 'tables' / 'dept_clusters.csv')
print('Distribution des clusters :')
print(clusters['cluster'].value_counts())

profiles = pd.read_csv(OUT_DIR / 'tables' / 'cluster_profiles.csv')
print('\nProfils des clusters :')
print(profiles.to_string(index=False))


## 5. Scénarios 2027

On projette l'abstention nationale T1 en 2027 avec 3 hypothèses :
- **Baseline** : tendances démographiques actuelles maintenues
- **Désengagement des jeunes** : +5 points de pourcentage de jeunes qui s'abstiennent
- **Remobilisation** : −4 points grâce à une forte mobilisation


In [ ]:
scenarios = pd.read_csv(OUT_DIR / 'tables' / 'scenarios_2027.csv')
print('Projections 2027 :')
print(scenarios.to_string(index=False))

fig, ax = plt.subplots(figsize=(8,4))
colors_sc = ['#2a78d6','#e34948','#1baf7a']
bars = ax.bar(scenarios['scenario'], scenarios['abstention_nationale_pred (%)'],
              color=colors_sc, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, scenarios['abstention_nationale_pred (%)']):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.3, f'{val:.1f}%',
            ha='center', fontsize=10, fontweight='bold')
ax.set_title('Prédictions du taux d\'abstention T1 2027', fontsize=13)
ax.set_ylabel('Taux d\'abstention prédit (%)') ; ax.set_ylim(0,35)
ax.axhline(26.3, color='grey', linestyle=':', linewidth=1, label='Taux 2022 (26.3%)')
ax.legend() ; ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15, ha='right') ; plt.tight_layout() ; plt.show()


## 6. Conclusion et recommandations

### Ce que le modèle dit :
- La tendance structurelle haussière de l'abstention devrait se poursuivre
- **Scénario central 2027 : ~29% d'abstention au T1** (vs 26% en 2022)
- L'écart entre les scénarios est modeste (4 pp) : les évolutions démographiques sont lentes

### Limites à communiquer à l'oral :
- R² négatifs en LOYO = les features démographiques seules ne suffisent pas
- Le **contexte politique** (popularité des candidats, enjeux perçus) joue un rôle que les données disponibles ne capturent pas
- 6 années d'observation = trop peu pour une validation robuste

### Pistes d'amélioration pour des projets futurs :
- Intégrer des données socio-économiques (chômage, revenus, niveau d'éducation par département)
- Ajouter des sondages d'opinion comme features exogènes
- Modèles de panel (effets fixes département)
